# Tetra Star Tracker Experiment

This revised notebook uses `catalog.bin`, builds a cached four-star Tetra database, tests identification, and reports FOV/magnitude accuracy.


## Load Catalog


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.star_tracker_core import *

ensure_dirs()
catalog = load_catalog()
print(f"Catalog stars: {len(catalog):,}")
display(catalog.head(10))


## Catalog Plot


In [ ]:
catalog_plot_path = plot_catalog(catalog, "tetra_catalog_map.png")
print(catalog_plot_path)


## Build Database


In [ ]:
MAX_STARS_QUERY = 12

tetra_db = build_tetra_database(
    catalog,
    fov_deg=8.0,
    mag_limit=6.5,
    max_tetras_per_anchor=20,
)
print(f"Tetra database rows: {len(tetra_db):,}")
display(tetra_db.head())

def identify(obs_ra, obs_dec, fov_deg, mag_limit, max_stars_query):
    return identify_tetra(
        catalog,
        tetra_db,
        obs_ra,
        obs_dec,
        fov_deg,
        mag_limit,
        max_stars_query=max_stars_query,
    )


## Single Identification Test


In [ ]:
single_result = identify(
    obs_ra=120.0,
    obs_dec=15.0,
    fov_deg=12.0,
    mag_limit=6.5,
    max_stars_query=MAX_STARS_QUERY,
)
print(single_result["outcome"])
print("stars in field:", single_result["n_stars"])
print("matched HR IDs:", single_result["matched_ids"])
print("score:", single_result["score"])
single_plot_path = plot_single_result(single_result, "tetra_single_result.png", "Tetra single-field result")
print(single_plot_path)


## Batch Identification Test


In [ ]:
batch_results = run_batch(
    identify,
    BatchConfig(samples=20, fov_deg=10.0, mag_limit=6.5, max_stars_query=MAX_STARS_QUERY),
)
batch_summary = summarize_results(batch_results)
display(batch_results.head(10))
batch_summary


## FOV and Magnitude Accuracy Matrix


In [ ]:
FOV_VALUES = [8.0, 12.0, 18.0]
MAG_LIMITS = [5.0, 6.0, 6.5, None]
MATRIX_SAMPLES = 8

confusion_df, summary_df = run_confusion_matrix(
    identify,
    fov_values=FOV_VALUES,
    mag_limits=MAG_LIMITS,
    samples=MATRIX_SAMPLES,
    max_stars_query=MAX_STARS_QUERY,
)
confusion_path = plot_confusion_matrix(confusion_df, "Tetra accuracy", "tetra_confusion_matrix.png")

display(confusion_df)
display(summary_df)
print(confusion_path)


## Findings


In [ ]:
report = findings(summary_df)
print(report)
